In [3]:
import os
import base64
import json
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd


In [20]:
def gemma(message):
    load_dotenv()
    client = OpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    response = client.chat.completions.create(
      model="gemma-3-4b-it",
      messages=message,
      stream=True
    )

    resp = ""
    for chunk in response:
        out_token = chunk.choices[0].delta.content
        if out_token is not None: 
            resp+=out_token
            print(out_token, end="")
    print("\n")
    return resp

In [21]:
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')


In [8]:
import csv
headers = ["chunk_timestamps","llm_explanation","keywords","subtitle"]
try:
    with open("output.csv", 'w', newline='') as file:
        writer = csv.writer(file)

        # Write the headers
        writer.writerow(headers)

    print(f"output.csv created successfully with list data.")
except IOError as e:
    print(f"Error writing to output.csv: {e}")


output.csv created successfully with list data.


In [52]:
def save_seg_out(seg_out):

    file_path = "output.jsonl"
    with open(file_path, 'a', encoding='utf-8') as f:
        f.write(seg_out+"\n")

In [ ]:
def msg(comb_inp_pth):
    frames_base = "/home/znyd/hacking/edu-cut/src/pre_processing/vid_frames/"
    prompt = {
           "type": "text",
           "text": """
          ### ROLE & GOAL ###

          You are an expert data processing engine specializing in semantic analysis. Your function is to analyze a sequence of pre-defined video clips of a long video and create a corresponding JSON object for each one. Each output object must be a comprehensive, self-contained summary structured for optimal retrieval from a vector database.

          ### CONTEXT & INPUT STRUCTURE ###

          I am providing you with the full content of an educational video, pre-processed and structured as a series of interleaved data chunks. You will receive the data for each 10-second segment sequentially: first, all the keyframe images for that segment, followed by a text block containing all the subtitles for that same segment. You will process all of these sequential chunks to understand the entire video.

          ### PRIMARY OBJECTIVE ###

          Your mission is to process **each 10-second segment** from the input and generate **one corresponding JSON summary object** for it.

          The most critical part of your output is the `llm_explanation` field within each object. This paragraph must be a dense, descriptive summary that synthesizes all the visual and spoken information from its corresponding 10-second clip, making the content fully understandable in isolation. This field is paramount as it will be used for vector embedding.

          Your final output must be a single, valid JSON array `[ { ... }, { ... } ]` containing one summary object for each input segment you were given.

          ### REQUIRED JSON STRUCTURE (for each object) ###

          {
            "chunk_timestamps": "string | The time range for this chunk, which you will infer from the input context (e.g., '0s-10s', '10s-20s').",
            "llm_explanation": "string | A dense, self-contained descriptive paragraph explaining the key concepts, steps, and visual information presented in this chunk. This text will be used for vector embedding.",
            "keywords": "array[string] | A list of few most relevant keywords that summarize the content of this chunk."
          }

          ---
          ---

          ### INPUT DATA ###

          (The user will now provide the interleaved data for each 10-second segment, starting with the frames and followed by the subtitles for that segment, repeated for the entire video clip.)
          """ 
        }
    messages = [
    {
      "role": "user",
      "content": [prompt,
        ],
    }
    ]

    with open(comb_inp_pth, 'r', encoding='utf-8') as f:
       loaded_json = json.load(f)
    print(len(loaded_json))


    for idx, seg in enumerate(loaded_json):
        time_stamps = seg['time_stamps']
        frames = seg['frames']
        subtitle = seg['subtitle']

        messages[0]["content"].append({
              "type": "text",
              "text": f"--- DATA FOR SEGMENT {time_stamps} ---",
            })

        #Adding image frames
        for frame in frames:
            # base64_image = encode_image(frames_base+frame)
            messages[0]["content"].append( {
              "type": "image_url",
              "image_url": { "url": f"data:image/png;base64,{frame}" },
            })

        #Adding subtitles for frames above
        subtitle_prompt = f"Subtitles for {time_stamps}:\n" 
        subtitle_prompt+="\n".join(subtitle)
        messages[0]["content"].append({
              "type": "text",
              "text": subtitle_prompt,
            })
        
        if (idx+1) % 6 == 0:
            print(messages)
            # save_resp = gemma(messages)[10:][:-5]+","
            # save_seg_out(save_resp)
            messages[0]['content'] = [prompt,] 
        elif idx+1 == len(loaded_json):
            print(messages)
            # save_resp = gemma(messages)[10:][:-5]+","
            # save_seg_out(save_resp) 
            messages[0]['content'] = [prompt,]

In [ ]:
msg("/home/znyd/hacking/edu-cut/src/pre_processing/combined_output.json")


In [ ]:
out_df = pd.read_csv('output.csv')
with open('output.json', 'r', encoding='utf-8') as f:
    out_load = json.load(f)

with open("/home/znyd/hacking/edu-cut/src/pre_processing/segment_subtitle.json", 'r', encoding='utf-8') as f:
    sub_data = json.load(f)

for idx, out_seg in enumerate(out_load):
    ts = out_seg['chunk_timestamps']
    llm_exp = out_seg['llm_explanation'] 
    keyword = out_seg['keywords']
    out_df.loc[idx, 'chunk_timestamps'] = ts
    out_df.loc[idx, 'llm_explanation'] = llm_exp
    out_df.loc[idx, 'keywords'] = keyword 
    out_df.loc[idx, 'subtitle'] = "\n".join(sub_data[ts])
out_df.to_csv('output.csv', index=False, quoting=csv.QUOTE_ALL)
print("output.csv successfully Done")

